# Budget processing routine

This notebook contains code to pre-process online budgets into grouped budget terms that are then saved to file (to be then read in by, for example, `Mixed_Layer_Temperature_Budget.ipynb`).

Note that the code here has largely been superseeded by scripts that do the processing work on PBS jobs, one year at a time.

These scripts are:
- `spawn_process_online_budget.py` - spawns the various online budget processing routines, one year at a time.
- `spawn_process_offline_monthly_budget.py` - spawns the offline monthly budget processing routine `process_offline_monthly_budget_year.sub`
- `spawn_process_offline_daily_budget.py` - spawns the offline daily budget processing routine `process_offline_daily_budget_month.sub`, one month at a time.

There are several different versions of the online budget processing script depending on whether one is considering the temperature or salinity budget, or standard or hat averaging.


In [ ]:
#Load required packages
%matplotlib inline
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
import pandas as pd
import cftime
from tqdm import tqdm

import cmocean as cm
import sys, os
import datetime

from dask.distributed import Client

In [ ]:
# Load workers:
client = Client(n_workers=4)
client

In [ ]:
# change directory to Figures/ subfolder for saving images
os.chdir('access-om2-analysis/access-om2-sst-budget/Figures')

# Load data

### Define paths, region to analyse and time period to analyse

In [ ]:
base = '/scratch/e14/rmh561/access-om2/archive/025deg_jra55_iaf_cycle6_online_mlt/'
output = 364 # 364 = 2017
#output = 365 # 365 = 2018
#output = 366 # 366 = 2019 - contains 3D daily budget diagnostics for quantifying correlation errors

tmp_folder = base + 'post_processed_diags/'

base2 = base + 'output%03d/ocean/' % output

# Subsample regions:
reg = [-270, -70, -60, 60] # Pacific

# Subsample time:
times = slice(None,None)
times_snap = slice(None,None) # Note; this must be 1 more than times.

chunks2D = {'time':1,'yt_ocean':216,'xt_ocean':240}
chunks3D = {'time':1,'st_ocean':25,'yt_ocean':324,'xt_ocean':360}

### Load grid and standard variables

In [ ]:
ds_grid = xr.open_dataset(base2 + 'ocean_grid.nc',chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))#.isel(time=times)
rho0 = 1035.
Cp = 3992.10322329649

ds_day = xr.open_dataset(base2 + 'ocean_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_day = ds_day.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_day.time.values]})
ds_day.average_DT.data = ds_day.average_DT*np.timedelta64(1,'D')
ds_day = ds_day.sel(time=times)

### Load ml-binned budget variables and snapshots

In [ ]:
# Standard average daily budget diagnostics:
ds_day_budget = xr.open_dataset(base2 + 'ocean_budget_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))

# Falling average daily budget daignostics (while the name of the averaging is "risavg", in effect this is actually the falling average diagnostics):
ds_day_budget_falavg = xr.open_dataset(base2 + 'ocean_budget_daily_risavg.nc',decode_times=False).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))

# Fix time variable by decoding time by hand (see https://forum.access-hive.org.au/t/cftime-vs-datetime64-time-encoding-issues-with-access-om2-025-omip-2-run/4085);
ds_day_budget = ds_day_budget.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_day_budget.time.values]})
ds_day_budget_falavg = ds_day_budget_falavg.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_day_budget_falavg.time.values]})

# Fix average_DT by decoding by hand:
ds_day_budget.average_DT.data = ds_day_budget.average_DT*np.timedelta64(1,'D')
ds_day_budget_falavg.average_DT.data = ds_day_budget_falavg.average_DT*np.timedelta64(1,'D')

# Subselect time period:
ds_day_budget = ds_day_budget.sel(time=times)
ds_day_budget_falavg = ds_day_budget_falavg.sel(time=times)

In [ ]:
# Snapshots for standard average tendency computation:
ds_day_snapshot = xr.open_dataset(base2 + 'ocean_snapshot_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
# Add previous output for last element:
ds_day_snapshot_m1 = xr.open_dataset(base2.replace(str(output),str(output-1)) + 'ocean_snapshot_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_day_snapshot = xr.concat([ds_day_snapshot_m1.isel(time=-1),ds_day_snapshot],dim='time')

# Fix time variable by decoding time by hand (see https://forum.access-hive.org.au/t/cftime-vs-datetime64-time-encoding-issues-with-access-om2-025-omip-2-run/4085);
ds_day_snapshot = ds_day_snapshot.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_day_snapshot.time.values]})

# Fix average_DT by decoding by hand:
ds_day_snapshot = ds_day_snapshot.sel(time=times_snap)

# Compute climatologies for selected standard variables:

In [ ]:
outputs = np.arange(336,366)                                           # Define outputs to include in climatology
fields = {'ocean_snapshot_month.nc':'all',                             # Define variables to compute
          'ocean_month.nc':['temp_in_mld','salt_in_mld','mld']
         }

dest_post = '_output%03d-%03d.clim.nc' % (outputs[0],outputs[-1])      # postfix for files

In [ ]:
for file in fields.keys():
    print('Doing ' + file + '...')
    ds = xr.open_dataset(base + 'output%03d' % outputs[0] + '/ocean/' + file,decode_times=False)
    if fields[file] != 'all':
        ds = ds[fields[file]]
    ds.load()

    for output in tqdm(outputs[1:]):
        ds2 = xr.open_dataset(base + 'output%03d' % output + '/ocean/' + file,decode_times=False)
        if fields[file] != 'all':
            ds2 = ds2[fields[file]]
        ds2.load()
        ds2 = ds2.assign_coords({'time':ds.time})
        ds = ds + ds2

    ds = ds/len(outputs)
    ds.to_netcdf(tmp_folder + file.replace('.nc',dest_post))        

# Define functions to construct grouped Mixed-layer temperature budget terms

First we define the term groups

In [ ]:
bud_tendency = 'temp_tendency_in_mld_cor'
bud_var_grps = {'advection':['temp_advection_in_mld_cor',
                             'temp_submeso_in_mld',
                             'neutral_diffusion_in_mld_temp',
                             'neutral_gm_in_mld_temp',
                             'temp_vdiffuse_k33_in_mld'],
                'vert_mixing':['temp_nonlocal_KPP_in_mld',
                               'temp_vdiffuse_diff_cbt_in_mld'],
                'surface_flux':['temp_rivermix_in_mld',
                                'temp_vdiffuse_sbc_in_mld', 
                                'frazil_3d_in_mld',
                                'sfc_hflux_pme_in_mld_cor',
                                'temp_eta_smooth_in_mld_cor'], 
                'sw_pen':['sw_heat_in_mld']}
bud_var_extras = {'shortwave':['swflx_in_mld'],
                  'longwave':['lw_heat_in_mld'],
                  'sensible':['sens_heat_in_mld'],
                  'latent':['evap_heat_in_mld']}

Then define functions to compute the correction terms, do the grouping and compute the tendency and entrainment terms

In [ ]:
def compute_corrections(ds_day_budget):
    """
    Compute corrections to advection, surface mass flux and total tendency terms in ds_day_budget.

    For the P-E+R correction:
    -------------------------
    
    sfc_hflux_pme_in_mld              =     Ca*Qm/H*Cp                      (Wm-3)
    pme_river_times_temp_in_mld       =     CH*Qm/H                         (kg m-3 deg C s-1)
    sfc_hflux_pme_in_mld_cor          =     (Ca-CH)*Qm/H*Cp                 (Wm-3)

    For the advection correction:
    -----------------------------
    adv_cor1                          =     CH*divU/H                       (deg C s-1), where divU = Qm/rho0 + Ssmoother - deta/dt from the free-surface equation
    adv_cor2                          =     Cent*(1- H/(D+eta))*deta/dt/H   (deg C s-1)

    Produces the corrected terms sfc_hflux_pme_in_mld_cor, temp_eta_smooth_in_mld_cor, temp_advection_in_mld_cor and temp_tendency_in_mld_cor
    """

    # P-E+R correction: 
    pme_cor = -ds_day_budget['pme_river_times_temp_in_mld']/rho0
    ds_day_budget['sfc_hflux_pme_in_mld_cor'] = ds_day_budget['sfc_hflux_pme_in_mld'] + pme_cor*rho0*Cp

    # Eta-smoother correction:
    eta_smoother_cor = -ds_day_budget['eta_smoother_times_temp_in_mld']/rho0
    ds_day_budget['temp_eta_smooth_in_mld_cor'] = ds_day_budget['temp_eta_smooth_in_mld'] + eta_smoother_cor*rho0*Cp

    # Advection correction:
    adv_cor1 = (-ds_day_budget['eta_t_tendency_times_temp_in_mld'] + ds_day_budget['pme_river_times_temp_in_mld'] + ds_day_budget['eta_smoother_times_temp_in_mld'])/rho0
    adv_cor2 = ds_day_budget['s_surf_ent_temp']/rho0
    ds_day_budget['temp_advection_in_mld_cor'] = ds_day_budget['temp_advection_in_mld'] + adv_cor1*rho0*Cp  + adv_cor2*rho0*Cp

    # Total correction for entrainment-by-residual:
    ds_day_budget['temp_tendency_in_mld_cor'] = ds_day_budget['temp_tendency_in_mld']  + pme_cor*rho0*Cp + adv_cor1*rho0*Cp + adv_cor2*rho0*Cp + eta_smoother_cor*rho0*Cp

    return(ds_day_budget)

def mlt_budget_fixedh(ds_day_budget, do_extras = True):
    """
    Compute fixedh budget terms, including residual. Does not include entrainment or mlt tendency.
    if do_extras=True, also add extra budget terms (e.g. the components of the surface flux) from bud_var_extras.
    """
    # Extract variable sums in groups, dividing by rho0*Cp to convert to degC/second
    mlt_budget = (ds_day_budget[bud_tendency]/rho0/Cp).rename('fixedh_tendency').to_dataset()
    for var in bud_var_grps.keys():
        mlt_budget[var] = ds_day_budget[bud_var_grps[var][0]]/rho0/Cp
        if (len(bud_var_grps[var])>1):
            for raw_var in bud_var_grps[var][1:]:
                mlt_budget[var] += ds_day_budget[raw_var]/rho0/Cp
    
    # Compute residual for check:
    mlt_budget['residual'] = mlt_budget['fixedh_tendency'].copy(deep=True)
    for var in list(mlt_budget.data_vars):
        mlt_budget['residual'] -= mlt_budget[var]

    if do_extras:
        for var in bud_var_extras.keys():
            mlt_budget[var] = ds_day_budget[bud_var_extras[var][0]]/rho0/Cp
            if (len(bud_var_extras[var])>1):
                for raw_var in bud_var_extras[var][1:]:
                    mlt_budget[var] += ds_day_budget[raw_var]/rho0/Cp

    return(mlt_budget)

def compute_tendency_entrainment(mlt_budget,mlt_snap):
    """
    Compute tendency term from snapshots and entrainment by residual.
    """

    mlt_snap = mlt_snap.transpose(*mlt_budget['fixedh_tendency'].dims)

    # Compute mlt tendency by taking time derivative of snapshot mlt:
    mlt_budget['mlt_tendency'] = xr.zeros_like(mlt_budget.fixedh_tendency).copy(deep=True)
    mlt_budget['mlt_tendency'].data = mlt_snap.isel(time=slice(1,None)).values - mlt_snap.isel(time=slice(0,-1)).values
    # mlt_budget['mlt_tendency'] = mlt_budget['mlt_tendency']/(ds_day_budget.average_DT/np.timedelta64(1,'s')) # Note: Depending on time decoding this line may change
    DT = xr.zeros_like(mlt_budget['mlt_tendency'].time)
    DT.data = (mlt_snap.time.isel(time=slice(1,None)).values-mlt_snap.time.isel(time=slice(0,-1)).values)/np.timedelta64(1,'s')
    mlt_budget['mlt_tendency'] = mlt_budget['mlt_tendency']/DT

    # Compute entrainment by residual:
    mlt_budget['entrainment'] = -(mlt_budget['fixedh_tendency'] - mlt_budget['mlt_tendency'])

    return(mlt_budget)

# Define functions to construct grouped Mixed-layer salinity budget terms

As above, but for salinity

In [ ]:
bud_tendency = 'salt_tendency_in_mld_cor'
bud_var_grps = {'advection':['salt_advection_in_mld_cor',
                             'salt_submeso_in_mld',
                             'neutral_diffusion_in_mld_salt',
                             'neutral_gm_in_mld_salt',
                             'salt_vdiffuse_k33_in_mld'],
                'vert_mixing':['salt_nonlocal_KPP_in_mld',
                               'salt_vdiffuse_diff_cbt_in_mld'],
                'surface_flux':['salt_rivermix_in_mld',
                                'salt_vdiffuse_sbc_in_mld',       # This is the restoring term - if wanting to separate it out.
                                'pme_in_mld_cor',                 # THis is the surface freshwater flux term - the main one. It enters as a "correction" term in the language of the MLT budget described in the theory above and in the paper.
                                'salt_eta_smooth_in_mld_cor']}
bud_var_extras = {}

In [ ]:
def compute_corrections(ds_day_budget):
    """
    Compute corrections to advection and surface mass flux terms in ds_day_budget for salinity
    """

    # P-E+R correction:
    # pme_river_times_salt_in_mld = psu kg m -3 s-1
    pme_cor = -ds_day_budget['pme_river_times_salt_in_mld']/1000.
    ds_day_budget['pme_in_mld_cor'] = pme_cor

    # Eta-smoother correction:
    eta_smoother_cor = -ds_day_budget['eta_smoother_times_salt_in_mld']/1000.
    ds_day_budget['salt_eta_smooth_in_mld_cor'] = ds_day_budget['salt_eta_smooth_in_mld'] + eta_smoother_cor

    # Advection correction:
    adv_cor1 = -(ds_day_budget['eta_t_tendency_times_salt_in_mld']-ds_day_budget['pme_river_times_salt_in_mld'] - ds_day_budget['eta_smoother_times_salt_in_mld'])/1000.
    adv_cor2 = ds_day_budget['s_surf_ent_salt']/1000.
    ds_day_budget['salt_advection_in_mld_cor'] = ds_day_budget['salt_advection_in_mld'] + adv_cor1 + adv_cor2

    # Total correction for residual:
    ds_day_budget['salt_tendency_in_mld_cor'] = ds_day_budget['salt_tendency_in_mld']  + pme_cor + adv_cor1 + adv_cor2 + eta_smoother_cor

    return(ds_day_budget)

def mlt_budget_fixedh(ds_day_budget, do_extras = True):
    """
    Compute fixedh budget terms, including residual. Does not include entrainment or mlt tendency.
    if do_extras=True, also add extra budget terms (e.g. the components of the surface flux) from bud_var_extras.
    """
    # Extract variable sums in groups, dividing by rho0 and multiplying by 1000 to convert to psu/sec
    mlt_budget = (ds_day_budget[bud_tendency]/rho0*1000.).rename('fixedh_tendency').to_dataset()
    for var in bud_var_grps.keys():
        mlt_budget[var] = ds_day_budget[bud_var_grps[var][0]]/rho0*1000.
        if (len(bud_var_grps[var])>1):
            for raw_var in bud_var_grps[var][1:]:
                mlt_budget[var] += ds_day_budget[raw_var]/rho0*1000.
    
    # Compute residual for check:
    mlt_budget['residual'] = mlt_budget['fixedh_tendency'].copy(deep=True)
    for var in list(mlt_budget.data_vars):
        mlt_budget['residual'] -= mlt_budget[var]

    if do_extras:
        for var in bud_var_extras.keys():
            mlt_budget[var] = ds_day_budget[bud_var_extras[var][0]]/rho0*1000.
            if (len(bud_var_extras[var])>1):
                for raw_var in bud_var_extras[var][1:]:
                    mlt_budget[var] += ds_day_budget[raw_var]/rho0*1000.

    return(mlt_budget)

def compute_tendency_entrainment(mlt_budget,mlt_snap):
    """
    Compute tendency term from snapshots and entrainment by residual.
    """

    mlt_snap = mlt_snap.transpose(*mlt_budget['fixedh_tendency'].dims)

    # Compute mlt tendency by taking time derivative of snapshot mlt:
    mlt_budget['mlt_tendency'] = xr.zeros_like(mlt_budget.fixedh_tendency).copy(deep=True)
    mlt_budget['mlt_tendency'].data = mlt_snap.isel(time=slice(1,None)).values - mlt_snap.isel(time=slice(0,-1)).values
    # mlt_budget['mlt_tendency'] = mlt_budget['mlt_tendency']/(ds_day_budget.average_DT/np.timedelta64(1,'s')) # Note: Depending on time decoding this line may change
    DT = xr.zeros_like(mlt_budget['mlt_tendency'].time)
    DT.data = (mlt_snap.time.isel(time=slice(1,None)).values-mlt_snap.time.isel(time=slice(0,-1)).values)/np.timedelta64(1,'s')
    mlt_budget['mlt_tendency'] = mlt_budget['mlt_tendency']/DT

    # Compute entrainment by residual:
    mlt_budget['entrainment'] = -(mlt_budget['fixedh_tendency'] - mlt_budget['mlt_tendency'])

    return(mlt_budget)

# Compute budget terms

Note: These cells are just for testing, if you're working with pre-computed budget terms, none of this code should be needed. 

Pre-computation of grouped terms is best done with PBS jobs using `spawn_process_online_budget.py`

## daily budget, standard averaging

In [ ]:
# Compute in one go:
ds_day_budget = compute_corrections(ds_day_budget)
mlt_budget_stavg_daily = mlt_budget_fixedh(ds_day_budget)
mlt_budget_stavg_daily = compute_tendency_entrainment(mlt_budget_stavg_daily,ds_day_snapshot.temp_in_mld/rho0)
mlt_budget_stavg_daily.load();

In [ ]:
# Compute in blocks (e.g. if doing a whole year):

ds_day_budget = compute_corrections(ds_day_budget)

# NOTE: THIS still seems way slower than it should be... Something simple might make it faster...
bs = 30; tl = len(ds_day_budget.time)
blocks = [range(tl)[x*bs:(x+1)*bs] for x in range(int(np.ceil(tl/bs)))]
blocks_snap = [range(tl+1)[x*bs:(x+1)*bs + 1] for x in range(int(np.ceil(tl/bs)))]

mlt_budget_stavg_daily_uncat = []
for i in tqdm(range(len(blocks))):
    bud = mlt_budget_fixedh(ds_day_budget.isel(time=blocks[i]))
    bud = compute_tendency_entrainment(bud,ds_day_snapshot.temp_in_mld.isel(time=blocks_snap[i])/rho0)
    mlt_budget_stavg_daily_uncat.append(bud.load())
mlt_budget_stavg_daily = xr.concat(mlt_budget_stavg_daily_uncat,dim='time')

In [ ]:
# Save to file if desired:
mlt_budget_stavg_daily.to_netcdf(tmp_folder + 'mlt_budget_stavg_daily_online_output%03d.nc' % output)

## Compute monthly difference budget, standard averaging
`mlt_budget_stavg_monthly` is simply the time integral of of the daily `mlt_budget_stavg_daily` budget. The resulting array contains the temperature difference induced by each term (or the temperature difference itself, for mlt\_tendency) across the month. Units are degC.

In [ ]:
mlt_budget_stavg_monthly = (mlt_budget_stavg_daily*(ds_day.average_DT/np.timedelta64(1,'s'))).resample(time='1ME').sum()
mlt_budget_stavg_monthly['time'] = mlt_budget_stavg_daily.time.resample(time='1ME').mean() # Centre time in the middle of the month.

In [ ]:
ds = mlt_budget_stavg_monthly.entrainment.load()

In [ ]:
# Another res check:
ds = mlt_budget_stavg_daily['mlt_tendency'] - mlt_budget_stavg_daily['advection'] - mlt_budget_stavg_daily['surface_flux']  - mlt_budget_stavg_daily['vert_mixing']   - mlt_budget_stavg_daily['sw_pen']    - mlt_budget_stavg_daily['entrainment']

In [ ]:
(ds.isel(time=-10)*86400).plot()

In [ ]:
fig, axes  = plt.subplots(nrows=3,ncols=4,figsize=(20,10))
for i, var in enumerate(mlt_budget_stavg_daily.data_vars):
    (mlt_budget_stavg_daily[var]*86400).isel(time=-10).plot(ax=axes.reshape(-1)[i],vmin=-.5,vmax=.5,cmap='RdBu_r')
    axes.reshape(-1)[i].set_title(var)
    #ds.isel(time=mn).plot(ax=axes.reshape(-1)[mn],vmin=-2.,vmax=2.,cmap='RdBu_r')
plt.tight_layout()

## Compute daily budget, hat averaging

Compute the hat-averaged daily budget. This budget (containing the same fields as `mlt_budget_stavg_daily`) corresponds to the tendency of the *daily-averaged* mixed layer temperature. E.g., mlt\_tendency in `mlt_budget_hatavg_daily` is the difference in daily-averaged temperature between the day after and the day before the time stamp (with the time stamp being at 0Z inbetween the two days), divided by the number of seconds in the day (units degC/second). The other terms are the budget contributions to this tendency. 

In [ ]:
def hat_average(st_avg,fal_avg_raw,average_DT):
    """
    Compute hat average from standard and falling average for a particular field
    """
    fal_avg = fal_avg_raw/(average_DT/np.timedelta64(1,'s')) # This line fixes a bug in the normalization of the daily falling average diagnostics
                                                             # take mean by dividing by averaging period. We do this here to avoid loading all the variables just to do this fix.
    ris_avg = st_avg - fal_avg                               # Rising = standard - falling
    hat_avg = ris_avg.isel(time=slice(0,-1)).values +  fal_avg.isel(time=slice(1,None)) # Hat = rising over first day - falling over second day              
                                                                                        # Note: Dealing with time is done outside this function
    return(hat_avg)

In [ ]:
# Template variable (note that hat average tendencies lie on snapshot (1:end-1) time:
mlt_budget_hatavg_daily = np.nan*xr.zeros_like(ds_day_snapshot.temp_in_mld.isel(time=slice(1,-1)).transpose(*ds_day_budget['temp_tendency_in_mld'].dims)) 

# Compute hat average fixedh tendency:
mlt_budget_hatavg_daily.data = hat_average(ds_day_budget['temp_tendency_in_mld'],ds_day_budget_falavg['temp_tendency_in_mld'],ds_day_budget_falavg.average_DT)/rho0/Cp

# Make a dataset:
mlt_budget_hatavg_daily = mlt_budget_hatavg_daily.rename('fixedh_tendency').to_dataset()

# Do other variables:
for var in bud_var_grps.keys():
    mlt_budget_hatavg_daily[var] = xr.zeros_like(mlt_budget_hatavg_daily['fixedh_tendency']).copy(deep=True)
    mlt_budget_hatavg_daily[var].data = hat_average(ds_day_budget[bud_var_grps[var][0]],ds_day_budget_falavg[bud_var_grps[var][0]],ds_day_budget_falavg.average_DT)/rho0/Cp
    if (len(bud_var_grps[var])>1):
        for raw_var in bud_var_grps[var][1:]:
            mlt_budget_hatavg_daily[var].data += hat_average(ds_day_budget[raw_var],ds_day_budget_falavg[raw_var],ds_day_budget_falavg.average_DT)/rho0/Cp

# Compute residual for check:
mlt_budget_hatavg_daily['residual'] = mlt_budget_hatavg_daily['fixedh_tendency'].copy(deep=True)
for var in list(mlt_budget_hatavg_daily.data_vars):
    mlt_budget_hatavg_daily['residual'] -= mlt_budget_hatavg_daily[var]

# Compute mlt tendency:
mlt = (ds_day.temp_in_mld/rho0).transpose(*mlt_budget_hatavg_daily['fixedh_tendency'].dims)
mlt_budget_hatavg_daily['mlt_tendency'] = xr.zeros_like(mlt_budget_hatavg_daily['fixedh_tendency']).copy(deep=True)
mlt_budget_hatavg_daily['mlt_tendency'].data = mlt.isel(time=slice(1,None)).values - mlt.isel(time=slice(0,-1)).values
mlt_budget_hatavg_daily['mlt_tendency'] = mlt_budget_hatavg_daily['mlt_tendency']/86400. # Note: this will only work for daily averaging, 
                                                                                           # as it assumes a 86400 time difference between 
                                                                                           # the centre of the two time-averaged periods (day before to day after)

# Entrainment term by residual:
mlt_budget_hatavg_daily['entrainment'] = -(mlt_budget_hatavg_daily['fixedh_tendency'] - mlt_budget_hatavg_daily['mlt_tendency'])

## Compute monthly difference budget, hat averaging

This section computes the monthly difference budgets (e.g. as above, contributions to the temperature differences across individual months) for standard averaging and hat averaging from the daily budgets. It then defines a function `monthly\_hat\_average` that takes these budgets as inputs, along with the monthly-averaged mixed layer temperature and two month indexes of interest, and outputs the contributions to the budget that govern the difference between the monthly-averaged mixed layer temperature of those two months.

Note: Some code and calculations here are repeated from the daily averaging performed above for conveninence.

In [ ]:
# Daily standard average budget (duplicated from above):
mlt_budget_stavg = mlt_budget_fixedh(ds_day_budget)

# Multiply by Delta t for differences:
mlt_budget_stavg = mlt_budget_stavg*(ds_day_budget.average_DT/np.timedelta64(1,'s'))

In [ ]:
# Daily falling average budget:
mlt_budget_falavg = mlt_budget_fixedh(ds_day_budget_falavg)

In [ ]:
# Daily rising average:
mlt_budget_risavg = mlt_budget_stavg - mlt_budget_falavg

In [ ]:
# Define n-1 DataArrays for the months:
n_minus_1 = xr.DataArray(data=[x-1 for x in mlt_budget_risavg.time.dt.day.values],dims=['time'],coords={'time':mlt_budget_stavg.time})

# Monthly rising average:
mlt_budget_risavg_monthly = mlt_budget_risavg.resample(time='1ME').mean() + (mlt_budget_stavg*n_minus_1).resample(time='1ME').mean()

# Monthly standard average:
mlt_budget_stavg_monthly = mlt_budget_stavg.resample(time='1ME').sum()

In [ ]:
# Define function to compute full budget given two months of interest:
def monthly_hat_difference(mlt_budget_risavg_monthly,mlt_budget_stavg_monthly,mlt_monthly,month1_index,month2_index):
    """
    Compute hat difference mlt budget from month 1 to month 2. 

    Inputs:
    - mlt_budget_risavg_monthly: The monthly difference budget, rising average
    - mlt_budget_stavg_monthly: The monthly difference budget, standard average
    - mlt_monthly: The monthly-average mixed layer temperature
    - month1_index: The index of the first month
    - month2_index: The index of the second month
    """

    # List of variables:
    vars = list(mlt_budget_risavg_monthly.data_vars)

    # Interim months:
    monthM_index = np.arange(month1_index+1,month2_index,1)
    
    # Compute mlt difference as "mlt_tendency":
    mlt_budget_hat_diff = (mlt_monthly.isel(time=month2_index) - mlt_monthly.isel(time=month1_index)).rename('mlt_tendency').to_dataset()

    # Compute falling difference as difference between other budgets:
    mlt_budget_falavg_monthly = mlt_budget_stavg_monthly - mlt_budget_risavg_monthly

    # Compute budget terms:
    for var in vars:
        mlt_budget_hat_diff[var] = mlt_budget_risavg_monthly[var].isel(time=month1_index) + mlt_budget_stavg[var].isel(time=monthM_index).sum('time') + mlt_budget_falavg_monthly[var].isel(time=month2_index)

    # Compute entrainment by residual:
    mlt_budget_hat_diff['entrainment'] = -(mlt_budget_hat_diff['fixedh_tendency'] - mlt_budget_hat_diff['mlt_tendency'])

    return(mlt_budget_hat_diff)